In [1]:
import pandas as pd

df = pd.read_csv("cleaned_ebay_deals.csv")

print("First 5 rows:")
print(df.head())

print("\nDataset shape:")
print(df.shape)

print("\nMissing values per column:")
print(df.isnull().sum())

First 5 rows:
             timestamp                                              title  \
0  2026-03-10 12:12:58  Lenovo M920q TINY / i5 8th / PICK YOUR RAM / P...   
1  2026-03-10 12:12:58  Sony Xperia 10 IV 5G XQ-CC54 128GB 6.0" Androi...   
2  2026-03-10 12:12:58  Nokia 1209 - Midnight Blue (Unlocked) Cellular...   
3  2026-03-10 12:12:58  Cartine Rizla Liquorice Corte – Gusto Naturale...   
4  2026-03-10 12:12:58  Honor Magic V 5G 512GB Google Android 12 Snapd...   

    price  original_price                   shipping  \
0  221.00          221.00  Shipping info unavailable   
1  171.58          171.58  Shipping info unavailable   
2   36.97           40.18  Shipping info unavailable   
3   35.94           35.94  Shipping info unavailable   
4  540.76          540.76  Shipping info unavailable   

                                            item_url  discount_percentage  
0  https://www.ebay.com/itm/256232862651?_trkparm...                 0.00  
1  https://www.ebay.com/itm/197837

In [2]:
df["price"] = pd.to_numeric(df["price"], errors="coerce")
df["original_price"] = pd.to_numeric(df["original_price"], errors="coerce")
df["discount_percentage"] = pd.to_numeric(df["discount_percentage"], errors="coerce")

In [3]:
df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
df["title"] = df["title"].fillna("").astype(str)

shipping_num = df["shipping"].astype(str).str.extract(r'(\d+\.?\d*)')[0]
shipping_num = pd.to_numeric(shipping_num, errors="coerce").fillna(0)

df["effective_price"] = df["price"] + shipping_num
df["discount_ratio"] = (df["original_price"] - df["price"]) / df["original_price"]

df["title_len"] = df["title"].str.len()
df["title_word_count"] = df["title"].str.split().str.len()

df["has_new"] = df["title"].str.contains("new", case=False, na=False).astype(int)
df["has_used"] = df["title"].str.contains("used", case=False, na=False).astype(int)
df["has_refurbished"] = df["title"].str.contains("refurbished", case=False, na=False).astype(int)
df["has_bundle"] = df["title"].str.contains("bundle", case=False, na=False).astype(int)
df["has_case"] = df["title"].str.contains("case", case=False, na=False).astype(int)
df["has_charger"] = df["title"].str.contains("charger", case=False, na=False).astype(int)

df["hour"] = df["timestamp"].dt.hour
df["weekday"] = df["timestamp"].dt.weekday

df.to_csv("ebay_features.csv", index=False)

In [4]:
from sklearn.model_selection import train_test_split

df["high_discount"] = (df["discount_percentage"] > 20).astype(int)

baseline_features = ["price", "original_price"]

extended_features = [
    "price", "original_price",
    "effective_price", "discount_ratio",
    "title_len", "title_word_count",
    "has_new", "has_used", "has_refurbished",
    "has_bundle", "has_case", "has_charger",
    "hour", "weekday"
]

X = df[extended_features]
y = df["high_discount"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("\nTarget distribution:\n", y.value_counts())

X_train shape: (67, 14)
X_test shape: (17, 14)

Target distribution:
 high_discount
0    81
1     3
Name: count, dtype: int64


In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

baseline_features = ["price", "original_price"]

extended_features = [
    "price", "original_price",
    "effective_price", "discount_ratio",
    "title_len", "title_word_count",
    "has_new", "has_used", "has_refurbished",
    "has_bundle", "has_case", "has_charger",
    "hour", "weekday"
]

X_baseline = df[baseline_features]
X_extended = df[extended_features]
y = df["high_discount"]

X_train_base, X_test_base, y_train, y_test = train_test_split(
    X_baseline, y, test_size=0.2, random_state=42, stratify=y
)

X_train_ext, X_test_ext, y_train, y_test = train_test_split(
    X_extended, y, test_size=0.2, random_state=42, stratify=y
)

models = {
    "Logistic Regression (Baseline)": LogisticRegression(max_iter=1000),
    "Logistic Regression + Feature Engineering": LogisticRegression(max_iter=1000),
    "Decision Tree Classifier": DecisionTreeClassifier(random_state=42)
}

results_text = ""

baseline_model = models["Logistic Regression (Baseline)"]
baseline_model.fit(X_train_base, y_train)
y_pred_base = baseline_model.predict(X_test_base)

acc = accuracy_score(y_test, y_pred_base)
prec = precision_score(y_test, y_pred_base, zero_division=0)
rec = recall_score(y_test, y_pred_base, zero_division=0)
f1 = f1_score(y_test, y_pred_base, zero_division=0)
cm = confusion_matrix(y_test, y_pred_base)

results_text += "Logistic Regression (Baseline)\n"
results_text += f"Accuracy: {acc}\n"
results_text += f"Precision: {prec}\n"
results_text += f"Recall: {rec}\n"
results_text += f"F1 Score: {f1}\n"
results_text += f"Confusion Matrix:\n{cm}\n\n"

feature_model = models["Logistic Regression + Feature Engineering"]
feature_model.fit(X_train_ext, y_train)
y_pred_ext = feature_model.predict(X_test_ext)

acc = accuracy_score(y_test, y_pred_ext)
prec = precision_score(y_test, y_pred_ext, zero_division=0)
rec = recall_score(y_test, y_pred_ext, zero_division=0)
f1 = f1_score(y_test, y_pred_ext, zero_division=0)
cm = confusion_matrix(y_test, y_pred_ext)

results_text += "Logistic Regression + Feature Engineering\n"
results_text += f"Accuracy: {acc}\n"
results_text += f"Precision: {prec}\n"
results_text += f"Recall: {rec}\n"
results_text += f"F1 Score: {f1}\n"
results_text += f"Confusion Matrix:\n{cm}\n\n"

tree_model = models["Decision Tree Classifier"]
tree_model.fit(X_train_ext, y_train)
y_pred_tree = tree_model.predict(X_test_ext)

acc = accuracy_score(y_test, y_pred_tree)
prec = precision_score(y_test, y_pred_tree, zero_division=0)
rec = recall_score(y_test, y_pred_tree, zero_division=0)
f1 = f1_score(y_test, y_pred_tree, zero_division=0)
cm = confusion_matrix(y_test, y_pred_tree)


results_text += "Decision Tree Classifier\n"
results_text += f"Accuracy: {acc}\n"
results_text += f"Precision: {prec}\n"
results_text += f"Recall: {rec}\n"
results_text += f"F1 Score: {f1}\n"
results_text += f"Confusion Matrix:\n{cm}\n"

with open("model_results.txt", "w") as f:
    f.write(results_text)

In [6]:
majority = df[df["high_discount"] == 0]
minority = df[df["high_discount"] == 1]

balanced = pd.concat([
    majority.sample(len(minority), random_state=42),
    minority
])

X_bal = balanced[extended_features]
y_bal = balanced["high_discount"]

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score

X_train_bal, X_test_bal, y_train_bal, y_test_bal = train_test_split(
    X_bal, y_bal,
    test_size=0.2,
    random_state=42,
    stratify=y_bal
)

model_bal = LogisticRegression(max_iter=1000)
model_bal.fit(X_train_bal, y_train_bal)

y_pred_bal = model_bal.predict(X_test_bal)

acc_bal = accuracy_score(y_test_bal, y_pred_bal)
prec_bal = precision_score(y_test_bal, y_pred_bal, zero_division=0)
rec_bal = recall_score(y_test_bal, y_pred_bal, zero_division=0)

with open("model_results.txt", "a") as f:
    f.write("\n=== After Handling Class Imbalance ===\n")
    f.write(f"Accuracy: {acc_bal}\n")
    f.write(f"Precision: {prec_bal}\n")
    f.write(f"Recall: {rec_bal}\n")